In [ ]:
import os
import json
import torch
import pandas as pd
import logging
import importlib
from pathlib import Path
import numpy as np
import random

import sys
PROJECT_DIR = "/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER"
SRC_DIR = str(Path(PROJECT_DIR) / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import multiomic_transformer.utils.data_formatter as data_formatter
import multiomic_transformer.utils.experiment_handler as experiment_handler

random.seed(1337)
np.random.seed(1337)
torch.manual_seed(1337)

# Training Data Preparation and Caching

The model uses cached per-chromosome data to help speed up training. Here, we will go through the process of building the necessary tensors and reference files to train the gene prediction model.

## Set up the TrainingDataFormatter

The training data formatter ensures that the correct data files and output directories exist and helps to keep everything standardized.

In [57]:
importlib.reload(data_formatter)

<module 'multiomic_transformer.utils.data_formatter' from '/gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/src/multiomic_transformer/utils/data_formatter.py'>

In [58]:
# Path to the project directory (same as Git repository root)
project_dir = Path(PROJECT_DIR)

# Path to the training output directory. Used to store the preprocessing config
output_dir = Path("/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments")

# Name of the dataset / experiment to run
dataset_name = "mESC_muon_data_caching_tutorial"

# Organism code for the dataset. Supports either "mm10" or "hg38"
organism_code = "mm10"

# List of samples in the training datset. 
# Each of these should have its own subdirectory in the processed data directory
sample_names = ["E7.5_rep1"]

# List of chromosomes. Used to split the data by chromsome for caching and training.
# Should be in the format "chr1", "chr2", etc. and should match the chromosome names in the processed data files.
chrom_list = [f"chr{i}" for i in range(1, 19)]

tdf = data_formatter.TrainingDataFormatter(
    project_dir=project_dir,
    dataset_name=dataset_name,
    organism_code=organism_code,
    sample_names=sample_names,
    chrom_list=chrom_list,
    output_dir=output_dir / dataset_name,
)

with open(tdf.output_dir / "file_paths.json", "r") as f:
    file_paths = json.load(f)
    


  - Found existing genome file: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
  - mm10.fa.gz is already BGZF; skipping transcode
  - Index already exists: mm10.fa.gz.fai, mm10.fa.gz.gzi
Genome ready: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/reference_genome/mm10/mm10.fa.gz
Found existing gene_info: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10
Found existing GTF: /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/genome_data/genome_annotation/mm10/Mus_musculus.GRCm39.115.gtf.gz
Map sizes: {'n_official': 143320, 'n_ens2sym': 77920, 'n_entrez2sym': 112254, 'n_alias2sym': 72569}


## Sample-Level Data Preparation

We first load the pseudobulk data from the Muon QC and preprocssing pipeline. The data files are loaded and the peak and gene names are standardized.

We load the pseudobulk data for the sample, which also creates a list of gene names. The `split_genes_into_tfs_and_tgs` function checks which of the genes in the gene name list overlap with an organism-specific [file of TF names from CIS-BP](https://mitra.stanford.edu/kundaje/marinovg/oak/various/motifs/CIS-BP/TF_Information_all_motifs.txt) to create `tfs` and `tgs` attributes.


In [59]:
sample_name = sample_names[0]

pseudobulk_rna_df = tdf.load_pseudobulk_rna_df(sample_name)

print(f"Number of genes: {len(tdf.genes):,}")

# Next the genes are classified as TFs or TGs by checking which genes in the data are in the reference TF list. 
tdf.tfs, tdf.tgs = tdf.split_genes_into_tfs_and_tgs(tdf.genes)
print(f"  - TFs: {len(tdf.tfs):,} (First 3: {tdf.tfs[:3]})")
print(f"  - TGs: {len(tdf.tgs):,} (First 3: {tdf.tgs[:3]})")

Number of genes: 23,882
  - TFs: 1,351 (First 3: ['2010315B03RIK', '2210418O10RIK', '2310033P09RIK'])
  - TGs: 22,515 (First 3: ['0610005C13RIK', '0610009E02RIK', '0610009L18RIK'])


The pseudobulk ATAC data is loaded, which also creates a list of peak names

In [60]:
pseudobulk_atac_df = tdf.load_pseudobulk_atac_df(sample_name)

print(f"Number of peaks: {len(tdf.peaks):,} (First 3: {tdf.peaks[:3]})")

Number of peaks: 199,885 (First 3: ['chr1:3035460-3036350', 'chr1:3044677-3045614', 'chr1:3062482-3063387'])


A BED file of the peak locations is created using `create_peak_bed_file` and the peak locations are stored as a dataframe in `tdf.peak_locs_df`.

In [61]:
tdf.create_peak_bed_file(pseudobulk_atac_df, sample_name)
display(tdf.peak_locs_df.head())

,chrom,start,end,strand,peak_id
0,chr1,3035460,3036350,.,chr1:3035460-3036350
1,chr1,3044677,3045614,.,chr1:3044677-3045614
2,chr1,3062482,3063387,.,chr1:3062482-3063387
3,chr1,3072145,3073018,.,chr1:3072145-3073018
4,chr1,3191389,3192286,.,chr1:3191389-3192286


### Calculate peak to gene distance

Next, we will use the peak bed file created when loading the peak pseudobulk dataset to calculate the distance between each peak and each gene within 1 Mb. This will create a bias tensor to help the model prioritize genes that are closer to peaks when training the peak-TG cross-attention portion of the model.

In [53]:
tdf.create_peak_to_tg_distance_file(
    sample_name=sample_name,
    max_peak_distance=1e6,
    distance_factor_scale=25000,
    force_recalculate=False,
    filter_to_nearest_gene=False,
)

Peak to TSS distance file already exists for sample E7.5_rep1 at /gpfs/Labs/Uzun/SCRIPTS/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/data/processed/mESC_muon_data_caching_tutorial/E7.5_rep1/peak_to_gene_dist.parquet. Skipping recalculation.


,peak_chr,peak_start,peak_end,peak_id,gene_chr,gene_start,gene_end,target_id,TSS_dist,TSS_dist_score
0,chr9,55148902,55149745,chr9:55148902-55149745,chr9,55149745,55149746,NRG4,0,1.000000e+00
1,chr7,27251562,27252441,chr7:27251562-27252441,chr7,27252441,27252442,PLD3,0,1.000000e+00
2,chr14,55918361,55919232,chr14:55918361-55919232,chr14,55919232,55919233,TINF2,0,1.000000e+00
3,chr11,57719406,57720297,chr11:57719406-57720297,chr11,57720297,57720298,GM32915,0,1.000000e+00
4,chr8,119436687,119437532,chr8:119436687-119437532,chr8,119437532,119437533,GM63757,0,1.000000e+00
...,...,...,...,...,...,...,...,...,...,...
50488308,chr1,24295262,24296008,chr1:24295262-24296008,chr1,23296008,23296009,GM53541,1000000,4.248354e-18
50488309,chr4,134886177,134887106,chr4:134886177-134887106,chr4,133887106,133887107,GM68734,1000000,4.248354e-18
50488310,chr5,151175820,151176702,chr5:151175820-151176702,chr5,150176702,150176703,GM72600,1000000,4.248354e-18
50488311,chr12,82274207,82275049,chr12:82274207-82275049,chr12,81275049,81275050,GM3693,1000000,4.248354e-18


## Chromosome-Specific Data Formatting


In [62]:
total_TG_pseudobulk_global, pseudobulk_chrom_dict = tdf.aggregate_pseudobulk_datasets(force_recalculate=False)
total_TG_pseudobulk_global.head()

  - Found existing global and per-chrom pseudobulk; loading...


,AAACAGCCAAACTCAT-1,AAACATGCAACCTGGT-1,AAACATGCAATGAATG-1,AAACATGCAATTATGC-1,AAACCAACATAATCGT-1,AAACCGAAGGACACTT-1,AAACCGCGTGAACAAA-1,AAACCGGCAACCTAAT-1,AAACCGGCAGAAATGC-1,AAACCGGCAGCACGAA-1,...,TTTGTCCCAGAATGAC-1,TTTGTCCCATAATTGC-1,TTTGTCTAGGTCGAGG-1,TTTGTGAAGTGAACCT-1,TTTGTGGCAATAATGG-1,TTTGTGGCAGCACCAT-1,TTTGTGGCAGCACGTT-1,TTTGTGGCATGAATAG-1,TTTGTTGGTCCTTCTC-1,TTTGTTGGTTAATGAC-1
0610005C13RIK,-0.056199,-0.233079,-0.202413,-0.194123,0.469641,0.961174,-0.221998,-0.191015,-0.166154,1.900074,...,0.197337,-0.179921,-0.237418,-0.234454,0.777808,-0.237418,-0.169386,-0.232935,-0.235122,-0.115639
0610009E02RIK,-0.159401,-0.159401,1.261940,-0.040533,-0.159401,-0.159401,0.774217,-0.040806,0.622838,-0.146429,...,0.004612,0.473802,-0.149608,-0.159401,-0.159401,-0.148427,0.466357,-0.147799,1.177287,0.307672
0610009L18RIK,-0.086806,-0.039594,0.008708,-0.089766,-0.080566,1.293913,-0.031008,-0.089766,-0.089766,0.062771,...,0.026476,-0.089766,0.393243,-0.025216,-0.089766,0.107134,0.003148,-0.089766,0.191206,-0.089766
0610012D04RIK,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,...,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256,-0.024256
0610025J13RIK,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,...,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054,-0.014054


In [64]:
tdf.create_chrom_files(total_TG_pseudobulk_global, pseudobulk_chrom_dict, force_recalculate=True, verbose=False)

  - Number of chromosomes: 18: ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18']
  - Processing chromosomes for dataset: mESC_muon_data_caching_tutorial
Creating chromosome-specific files for chr1
Creating chromosome-specific files for chr2
Creating chromosome-specific files for chr3
Creating chromosome-specific files for chr4
Creating chromosome-specific files for chr5
Creating chromosome-specific files for chr6
Creating chromosome-specific files for chr7
Creating chromosome-specific files for chr8
Creating chromosome-specific files for chr9
Creating chromosome-specific files for chr10
Creating chromosome-specific files for chr11
Creating chromosome-specific files for chr12
Creating chromosome-specific files for chr13
Creating chromosome-specific files for chr14
Creating chromosome-specific files for chr15
Creating chromosome-specific files for chr16
Creating chromosome-specific fil

We can check that all of the required cached files exist for the experiment.

In [65]:
missing_files = tdf.check_cached_file_exist()
print(f"Missing {len(missing_files)} cached files")

All required files are present.


Missing 0 cached files


# Testing how the data prep and caching interacts with the experiment_loader class

In [69]:
importlib.reload(experiment_handler)
exp = experiment_handler.ExperimentHandler(
    experiment_dir = "/gpfs/Labs/Uzun/DATA/PROJECTS/2024.SINGLE_CELL_GRN_INFERENCE.MOELLER/experiments/",
    experiment_name=dataset_name,
    model_num=1,
    silence_warnings=True,
)

exp.tf_names = tdf.tfs
exp.tg_names = tdf.tgs

exp.common_data_dir = tdf.file_paths["training_cache"]["common"]["dir"]
exp.data_cache_dir = tdf.file_paths["training_cache"]["dataset_dir"]
exp.chrom_ids = tdf.chrom_list

print("Creating dataset...")
exp.create_multichrom_dataset(max_cached=100,)

train_loader, val_loader, test_loader = exp.prepare_dataloader(
    batch_size=256,
    num_workers=8
)

print("Creating scalers...")
exp.create_scalers(train_loader)

print("Creating model")
exp.create_new_model()

print("Training model")
exp.train(
    train_loader=train_loader, 
    val_loader=val_loader, 
    num_epochs=2,
    max_batches=50,
    allow_overwrite=True,
    )



Creating dataset...


Creating scalers...


Model training directory already exists, incrementing model_num to 5


Creating model
Training model


Epoch 1/2 | Train Loss: 0.2316 | Val MSE: 0.1007 | R2 (Unscaled): -0.454 | R2 (Scaled): -0.454 | LR: 2.50e-04 | Time: 35.1s


Epoch 2/2 | Train Loss: 0.0981 | Val MSE: 0.0833 | R2 (Unscaled): -0.202 | R2 (Scaled): -0.202 | LR: 2.50e-04 | Time: 34.4s


MultiomicTransformer(
  (tf_identity_emb): Embedding(1351, 128)
  (tg_query_emb): Embedding(22515, 128)
  (window_downsampler): WindowDownsampler(
    (net): Sequential(
      (0): Conv1d(1, 32, kernel_size=(16,), stride=(16,))
      (1): GELU(approximate='none')
      (2): Conv1d(32, 1, kernel_size=(1,), stride=(1,))
    )
  )
  (tf_expr_dense_input_layer): Sequential(
    (0): Linear(in_features=1, out_features=512, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=False)
    (4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (atac_acc_dense_input_layer): Sequential(
    (0): Linear(in_features=1, out_features=512, bias=True)
    (1): SiLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=512, out_features=128, bias=False)
    (4): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (posenc): PositionalEmbedding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(


In [ ]:
exp.tf_scaler = exp.scalers["tf_scaler"]
exp.tg_scaler = exp.scalers["tg_scaler"]

exp.run_gradient_attribution(
    test_loader,
    max_batches=25,
    max_tgs_per_batch=128,
    )

In [ ]:

GROUND_TRUTH_DIR = Path(PROJECT_DIR) / "data" / "ground_truth_files"
gt_by_dataset_dict = {
        "ChIP-Atlas mESC": exp.load_ground_truth(GROUND_TRUTH_DIR / "chip_atlas_tf_peak_tg_dist.csv"),    
}
auroc_df = exp.calculate_auroc_all_sample_gts(exp.grn, gt_by_dataset_dict)     
auroc_df.head()       